In [1]:
import pandas as pd
import numpy as np
import requests
import json
from tqdm.notebook import tqdm
from datetime import datetime, timedelta
import threading
import sqlite3
from sqlalchemy import create_engine
from pathlib import Path

In [2]:
conn = sqlite3.connect('flight.db')
cursor = conn.cursor()

In [3]:
#flights table contains all flights
cursor.execute("""
    CREATE TABLE IF NOT EXISTS flights (
        flight_number INT,
        airport_id INT,
        scheduled_date DATETIME,
        cancelled BOOLEAN,
        weather_delay_time FLOAT,
        scheduled_date_round_utc DATETIME,
        month INT,
        day_of_week INT,
        PRIMARY KEY (flight_number, airport_id, scheduled_date),
        FOREIGN KEY (airport_id) REFERENCES airports(airport_id)
    )
""")
#contains coor data, i.e. coordinate data for each airport
cursor.execute("""
    CREATE TABLE IF NOT EXISTS airports(
        airport_id INT PRIMARY KEY,
        longitude FLOAT,
        latitude FLOAT
    )
""")
#contains weather data at each location for each instance of time
cursor.execute("""
    CREATE TABLE IF NOT EXISTS weather (
        longitude FLOAT,
        latitude FLOAT,
        date DATETIME,
        temp FLOAT,
        precip FLOAT,
        wind_speed_10 FLOAT,
        wind_dir_10 FLOAT,
        humidity FLOAT,
        pressure FLOAT,
        cloud_amt FLOAT,
        wind_speed_50 FLOAT,
        wind_dir_50 FLOAT,
        dew_point FLOAT,
        spread FLOAT,
        ice_risk BOOLEAN,
        PRIMARY KEY (longitude, latitude, date)
    )
""")
conn.commit()

In [4]:
#Take coordinate data and make sure only unique airports in coor
coor = pd.read_csv('T_MASTER_CORD.csv')
unique_airports = coor.groupby('AIRPORT_ID')['AIRPORT_SEQ_ID'].max()
coor = coor[coor['AIRPORT_SEQ_ID'].isin(unique_airports)]

#Take only needed fields to input to sql table
coor_input = coor.copy()[['AIRPORT_ID', 'LONGITUDE', 'LATITUDE']]
coor_input.columns = ['airport_id', 'longitude', 'latitude']
#create engine for uploading df to sql
database_connection_str = 'sqlite:///flight.db'
engine = create_engine(database_connection_str)

coor_input.to_sql(
    name='airports', 
    con=engine, 
    if_exists='replace', 
    index= False
)
conn.commit()

In [5]:
#functions to assist in datetime transforming
def to_hr(x):
    return int(x.zfill(4)[:2])
def to_min(x):
    return int(x.zfill(4)[2:])


In [6]:
#iterate through each monthly dataset in flight_data
folder = 'flight_data'
path = Path(folder) # Create a Path object
pbar = tqdm(total=(len(list(path.iterdir()))))
for month in path.iterdir():
    
    if month.suffix == '.csv':
        df = pd.read_csv(month)
        #assign utc difference to each flight record, and calculate actual scheduled departure
        utc = coor[['AIRPORT_ID', 'UTC_LOCAL_TIME_VARIATION']]
        df = pd.merge(df, utc, left_on ='ORIGIN_AIRPORT_ID', right_on ='AIRPORT_ID', how = 'left')
        df['Scheduled Departure'] = pd.to_datetime(df['FL_DATE'], format='%m/%d/%Y %I:%M:%S %p') + pd.to_timedelta(df['CRS_DEP_TIME'].astype(str).apply(to_hr), unit = 'h' ) + pd.to_timedelta(df['CRS_DEP_TIME'].astype(str).apply(to_min), unit = 'm' )
        #prepare necessary features to input to sql table

        input_cols = ['OP_CARRIER_FL_NUM', 'AIRPORT_ID', 'Scheduled Departure', 'CANCELLED', 'WEATHER_DELAY',  'MONTH', 'DAY_OF_WEEK']
        flight_input = df[(df['CANCELLATION_CODE'].isna()) | (df[ 'CANCELLATION_CODE'] == 'B')].copy()[input_cols]
        flight_input.columns = ['flight_number', 'airport_id', 'scheduled_date', 'cancelled','weather_delay_time', 'month', 'day_of_week']
        #create rounded date for weather api
        flight_input['scheduled_date_round_utc'] = pd.to_datetime(flight_input['scheduled_date']).dt.round('h').dt.strftime('%Y-%m-%d %H:%M:%S') 
        flight_input['scheduled_date'] =  flight_input['scheduled_date'] + pd.to_timedelta( df['UTC_LOCAL_TIME_VARIATION'].astype(int)//100 , unit = 'h')
        
        flight_input['cancelled'] = flight_input['cancelled'].astype(int)
        #fill missing values with 0 
        flight_input['weather_delay_time'] = flight_input['weather_delay_time'].fillna(0)
        #drop any possible duplicate flights
        flight_input = flight_input.drop_duplicates(subset=['airport_id', 'scheduled_date'])
        #upload data to table
        flight_input.to_sql(
            name='flights', 
            con=engine, 
            if_exists='append', 
            index= False          
        )
    pbar.update(1)
conn.commit()
pbar.close()


  0%|          | 0/13 [00:00<?, ?it/s]

In [8]:
#obtain all positions, with minimum and maximum date for time range 
query = """WITH date AS(
        SELECT airport_id, MIN(scheduled_date_round_utc) AS min_date, MAX(scheduled_date_round_utc) AS max_date FROM flights GROUP BY airport_id
        )
        SELECT a.latitude, a.longitude, d.min_date, d.max_date
        FROM date d LEFT JOIN airports a
        ON a.airport_id = d.airport_id        
        """
df = pd.read_sql(query, conn)

In [9]:
#prepare parameters for API calls
weather_params = "T2M,PRECTOTCORR,WS10M,RH2M,WD10M,PS,CLOUD_AMT,WS50M,WD50M,T2MDEW"
url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
df['min_date'] = pd.to_datetime(df['min_date'])
df['max_date'] = pd.to_datetime(df['max_date'])

In [10]:
#Chunk processing function for threading
def chunk_process(chunk, pbar):
    for row in chunk:
        #take each row of information and process
        lat, long, start, end = row
        key = str(lat) + ' ' + str(long)
        record = {}
        start = start- timedelta(1)
        end = end+ timedelta(1)
        params = {
            "parameters": weather_params,
            "community": "RE",
            "longitude": long,
            "latitude": lat,
            "start": start.strftime('%Y%m%d'),
            "end": end.strftime('%Y%m%d'),
            "format": "JSON",
            "time_standard": "UTC"
        }
        #call api for information and retrieve as json object
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()    

        #store all data in temporary dictionary object
        params_dict = data['properties']['parameter']
        times = list(params_dict['T2M'].keys())
        for time in times:
            record[time]= {
                "temp": params_dict['T2M'][time],
                "precip": params_dict['PRECTOTCORR'][time],
                "wind_speed_10": params_dict['WS10M'][time],
                "wind_dir_10": params_dict['WD10M'][time],
                "humidity": params_dict['RH2M'][time],
                "pressure": params_dict['PS'][time],
                "cloud_amt": params_dict['CLOUD_AMT'][time],
                "wind_speed_50":params_dict['WS50M'][time],
                "wind_dir_50": params_dict['WD50M'][time],
                "dew_point" : params_dict['T2MDEW'][time],
                "spread" : params_dict['T2M'][time] - params_dict['T2MDEW'][time],
                "ice_risk" : (params_dict['T2M'][time] < 2) & (params_dict['PRECTOTCORR'][time]>0)
        }
        #file lock makes sure only one thread can write at a time 
        with file_lock:
           with open(weather_file, 'a') as f:
               #jsonl files allows for appending as json format
               json.dump({key: record}, f)
               f.write('\n')
               
        pbar.update(1)

In [11]:
#break list of positions and date range into chunks and find weather data
weather_file = 'weather_data.jsonl'
file_lock = threading.Lock()
all_tasks = df.values
chunk_size = (len(all_tasks) // 4) + 1
chunks = [all_tasks[i:i + chunk_size] for i in range(0, len(all_tasks), chunk_size)]
pbar = tqdm(total=len(all_tasks))
threads = []
for chunk in chunks:
    t = threading.Thread(target=chunk_process, args=(chunk, pbar))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

pbar.close()


  0%|          | 0/352 [00:00<?, ?it/s]

In [12]:
#open weather data file
weather_data = {}
with open('weather_data.jsonl', 'r') as f:
    for line in f:
        weather_data.update(json.loads(line))

#upload weather data to sql table
for location in weather_data:
    lat, long = location.split()
    for time in weather_data[location]:
        data = weather_data[location][time]
        cursor.execute('INSERT OR IGNORE INTO weather VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)', 
                   (long, lat, datetime.strptime(time, '%Y%m%d%H') , data['temp'], data['precip'], 
                    data['wind_speed_10'], data['wind_dir_10'], data['humidity'],data['pressure'],data['cloud_amt'],
                   data['wind_speed_50'], data['wind_dir_50'], data['dew_point'], data['spread'], data['ice_risk']))
conn.commit()

/tmp/ipykernel_3761387/857280193.py:12: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('INSERT OR IGNORE INTO weather VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',


In [ ]:
conn.close()